# Suffix direct-MH SMC tail-probability validation

This notebook runs only `suffix_direct_mh_smc`. Every `(cap, terminal lambda, seed)` entry is a complete independent SMC run. Final weighted populations are combined with a deterministic-mixture balance heuristic and compared with the 4,803,396 official direct samples. Estimated SMC normalizers make the final estimator plug-in/SNIS, not exactly unbiased at finite particle count.

In [ ]:
# 1. Mount Google Drive for persistent production outputs.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repository, or safely fast-forward an existing clean checkout.
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

if (REPO_ROOT / '.git').is_dir():
    dirty = subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError('Existing checkout has local changes; review them before updating.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
print('Repository:', REPO_ROOT)

In [ ]:
# 3. Enter and validate the repository root.
import os
os.chdir(REPO_ROOT)
assert (REPO_ROOT / 'suffix_direct_mh_smc_experiment' / 'run_tail_validation.py').is_file()
assert (REPO_ROOT / 'results' / 'reference_direct_sampling_summary.json').is_file()
print('cwd =', Path.cwd())

In [ ]:
# 4. Install the minimal repository dependencies in this runtime.
%pip install -q -r requirements-colab.txt

In [ ]:
# 5. Inspect PyTorch, CUDA, and the official-reference limitation.
import json
import torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
reference = json.loads((REPO_ROOT / 'results' / 'reference_direct_sampling_summary.json').read_text())
print('Sampling setup:', reference['comparability']['sampling_setup_match'])
print('Strict observable match verified:', reference['comparability']['strict_observable_match_verified'])
print('Limitation:', reference['comparability']['reason_observable_match_unverified'])

In [ ]:
# 6. EDIT PARAMETERS HERE. This is the only experiment-configuration cell.
COMPONENTS = [
    {'cap': 6,  'lambda': -0.5},
    {'cap': 6,  'lambda': -1.0},
    {'cap': 6,  'lambda': -1.5},
    {'cap': 7,  'lambda': -1.0},
    {'cap': 7,  'lambda': -1.5},
    {'cap': 7,  'lambda': -2.0},
    {'cap': 8,  'lambda': -1.5},
    {'cap': 8,  'lambda': -2.0},
    {'cap': 8,  'lambda': -2.5},
    {'cap': 10, 'lambda': -2.0},
    {'cap': 10, 'lambda': -2.5},
    {'cap': 10, 'lambda': -3.0},
    {'cap': 10, 'lambda': -3.5},
]
TAIL_THRESHOLDS = '6,7,8,10'
LAMBDA_STEP = 0.25
SEED_GROUPS = [200]  # First production pass; later expand to [200, 300, 400].
NUM_PARTICLES = 256
MCMC_STEPS_PER_LEVEL = 10
ESS_RESAMPLING_THRESHOLD = 0.5
MAX_NEW_TOKENS = 100
SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SUFFIX_RHOS = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'
BATCH_SIZE = 32
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_tail_probability_validation')
SKIP_COMPLETED_RUNS = True
ALLOW_UNVERIFIED_OBSERVABLE_REFERENCE = False
DEVICE = 'cuda'
RUN_CACHE_AUDIT = True  # Reads Drive archives only; never generates samples.
RUN_SMOKE_TEST = True
RUN_PRODUCTION = False  # Set True only after the smoke test succeeds.

MODEL_NAME = 'roneneldan/TinyStories-8M'
TOKENIZER_NAME = 'EleutherAI/gpt-neo-125M'
PROMPT = 'Once upon a time, in a big forest, there lived a rhinoc'
assert len(SEED_GROUPS) == len(set(SEED_GROUPS))
print('Independent production runs:', len(COMPONENTS) * len(SEED_GROUPS))
print('Production output:', OUTPUT_ROOT)

In [ ]:
# 7. Build one command for both smoke and production configurations.
import json
import sys

def validation_command(*, components, seed_groups, particles, max_tokens, mcmc_steps,
                       suffix_sizes, suffix_rhos, batch_size, output_root, device,
                       smoke_mode=False, audit_only=False, allow_unverified_reference=False):
    command = [
        sys.executable, 'suffix_direct_mh_smc_experiment/run_tail_validation.py',
        '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
        '--prompt', PROMPT, '--device', device,
        '--components-json', json.dumps(components),
        '--seed-groups', ','.join(map(str, seed_groups)),
        '--thresholds', TAIL_THRESHOLDS, '--lambda-step', str(LAMBDA_STEP),
        '--num-particles', str(particles), '--max-new-tokens', str(max_tokens),
        '--mcmc-steps-per-level', str(mcmc_steps),
        '--ess-threshold', str(ESS_RESAMPLING_THRESHOLD),
        '--suffix-sizes', suffix_sizes, '--suffix-rhos', suffix_rhos,
        '--batch-size', str(batch_size), '--output-root', str(output_root),
    ]
    command.append('--skip-completed-runs' if SKIP_COMPLETED_RUNS else '--no-skip-completed-runs')
    if smoke_mode:
        command.append('--smoke-mode')
    if audit_only:
        command.append('--audit-only')
    if allow_unverified_reference:
        command.append('--allow-unverified-observable-reference')
    return command

In [ ]:
# 8. CPU integration smoke: it tests execution only, never estimator accuracy.
SMOKE_OUTPUT_ROOT = Path('/content/rea_tail_probability_validation_smoke')
if RUN_SMOKE_TEST:
    smoke_command = validation_command(
        components=COMPONENTS[:2], seed_groups=SEED_GROUPS[:1],
        particles=8, max_tokens=8, mcmc_steps=1,
        suffix_sizes='1,2,4,8', suffix_rhos='0.4,0.3,0.2,0.1',
        batch_size=8, output_root=SMOKE_OUTPUT_ROOT, device='cpu',
        smoke_mode=True, allow_unverified_reference=True,
    )
    subprocess.run(smoke_command, check=True)
    assert (SMOKE_OUTPUT_ROOT / 'comparison.csv').is_file()
    print('Smoke test passed:', SMOKE_OUTPUT_ROOT)
else:
    print('Smoke test skipped.')

In [ ]:
# 9. Audit the persistent cache, then optionally run missing production components.
if RUN_CACHE_AUDIT:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    audit_command = validation_command(
        components=COMPONENTS, seed_groups=SEED_GROUPS,
        particles=NUM_PARTICLES, max_tokens=MAX_NEW_TOKENS,
        mcmc_steps=MCMC_STEPS_PER_LEVEL, suffix_sizes=SUFFIX_SIZES,
        suffix_rhos=SUFFIX_RHOS, batch_size=BATCH_SIZE,
        output_root=OUTPUT_ROOT, device=DEVICE, audit_only=True,
        allow_unverified_reference=True,
    )
    subprocess.run(audit_command, check=True)
    print('Cache audit complete; no samples were generated.')

if RUN_PRODUCTION:
    if not ALLOW_UNVERIFIED_OBSERVABLE_REFERENCE:
        raise RuntimeError(
            'Production is blocked: exact official/current ARI equivalence is unverified. '
            'Set ALLOW_UNVERIFIED_OBSERVABLE_REFERENCE=True only for an explicitly approximate comparison.'
        )
    if DEVICE == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('DEVICE=cuda but this runtime has no CUDA GPU.')
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    production_command = validation_command(
        components=COMPONENTS, seed_groups=SEED_GROUPS,
        particles=NUM_PARTICLES, max_tokens=MAX_NEW_TOKENS,
        mcmc_steps=MCMC_STEPS_PER_LEVEL, suffix_sizes=SUFFIX_SIZES,
        suffix_rhos=SUFFIX_RHOS, batch_size=BATCH_SIZE,
        output_root=OUTPUT_ROOT, device=DEVICE,
        allow_unverified_reference=ALLOW_UNVERIFIED_OBSERVABLE_REFERENCE,
    )
    subprocess.run(production_command, check=True)
else:
    print('Production disabled. Set RUN_PRODUCTION=True after validating the smoke output.')

In [ ]:
# 10. Inspect cache classifications and load the saved comparison when available.
import pandas as pd
from IPython.display import display

ACTIVE_OUTPUT_ROOT = OUTPUT_ROOT if (RUN_PRODUCTION or RUN_CACHE_AUDIT) else SMOKE_OUTPUT_ROOT
audit_path = OUTPUT_ROOT / 'data_reuse_audit.csv'
audit_df = pd.read_csv(audit_path) if audit_path.is_file() else pd.DataFrame()
if not audit_df.empty:
    display(audit_df['classification'].value_counts().rename('archive_count').to_frame())
    print('Candidate archives found:', len(audit_df))
    print('Archives selected for mixture:', int(audit_df['selected_for_mixture'].sum()))
else:
    print('No candidate production archives found in:', OUTPUT_ROOT / 'component_runs')
comparison_path = ACTIVE_OUTPUT_ROOT / 'comparison.csv'
comparison_df = pd.read_csv(comparison_path) if comparison_path.is_file() and comparison_path.stat().st_size else pd.DataFrame()
aggregate_df = (comparison_df[comparison_df['row_type'] == 'aggregate'].copy()
                if 'row_type' in comparison_df else pd.DataFrame())
HAS_AUDITED_PRODUCTION_RESULTS = RUN_CACHE_AUDIT and not aggregate_df.empty
print('Active results:', ACTIVE_OUTPUT_ROOT)
print('Audited SMC-mixture comparison available:', HAS_AUDITED_PRODUCTION_RESULTS)

In [ ]:
# 11. Display verified cached production results or results generated in this session.
from IPython.display import Image, Markdown, display

if RUN_PRODUCTION or HAS_AUDITED_PRODUCTION_RESULTS:
    label = ('new production run' if RUN_PRODUCTION else 'audited cached production run')
    display(Markdown(f'**Showing {label}: suffix SMC mixture versus official direct sampling.**'))
    display(aggregate_df)
    if (ACTIVE_OUTPUT_ROOT / 'ari_boxplots.png').is_file():
        display(Image(filename=str(ACTIVE_OUTPUT_ROOT / 'ari_boxplots.png')))
    if (ACTIVE_OUTPUT_ROOT / 'probability_comparison.png').is_file():
        display(Image(filename=str(ACTIVE_OUTPUT_ROOT / 'probability_comparison.png')))
    display(Markdown((ACTIVE_OUTPUT_ROOT / 'REPORT.md').read_text(encoding='utf-8')))
else:
    display(Markdown('**No complete audited production result is available. Smoke estimates remain hidden because they do not validate accuracy.**'))